#### Run the black-box judge on Colab GPU with a Hugging Face model

In [1]:
import subprocess, time
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

In [2]:
# Judge model: Ministral 3 8B Instruct
JUDGE_MODEL = "hf.co/mistralai/Ministral-3-8B-Instruct-2512-GGUF:Q4_K_M"
!ollama pull {JUDGE_MODEL}


In [3]:
import os, zipfile
from google.colab import files
up = files.upload()
for name in up:
    zipfile.ZipFile(name).extractall(".")
!pip -q install pyyaml tqdm
os.environ["OLLAMA_API_KEY"] = "local"

In [4]:
ENDPOINT = "http://localhost:11434/v1/chat/completions"
# (data tag, output name). Tag "" is Qwen; the others need data/multi_turn_gen_<tag>/ in the zip.
RUNS = [("", "qwen2.5-3b"), ("smollm2-1.7b", "smollm2-1.7b"), ("olmo-2-1b", "olmo-2-1b")]
for tag, name in RUNS:
    folder = f"data/multi_turn_gen_{tag}" if tag else "data/multi_turn_gen"
    if not os.path.isdir(folder):
        print("skip", name, "(no", folder + ")"); continue
    tag_arg = f"--data-tag {tag}" if tag else ""
    !PYTHONPATH=src python scripts/run_judge.py {tag_arg} --judge-model {JUDGE_MODEL} --endpoint {ENDPOINT} --api-key-env OLLAMA_API_KEY --min-interval 0 --out results/tables/judge_hf_{name}.jsonl

Scoring judge turns: 100% 400/400 [33:05<00:00,  4.96s/turn]
INFO Judge turns written to results/tables/judge_hf_qwen2.5-3b.jsonl
INFO Prompt SHA-256: 047a96d300b1d7d3bf97e302ea3c7285ea94871cbd2e887d6b148ca6ccea8da4
INFO Freeze this hash in configs/experiment.yaml before rerunning.
Scoring judge turns: 100% 400/400 [31:24<00:00,  4.71s/turn]
INFO Judge turns written to results/tables/judge_hf_smollm2-1.7b.jsonl
INFO Prompt SHA-256: 047a96d300b1d7d3bf97e302ea3c7285ea94871cbd2e887d6b148ca6ccea8da4
INFO Freeze this hash in configs/experiment.yaml before rerunning.
Scoring judge turns: 100% 400/400 [35:02<00:00,  5.26s/turn]
INFO Judge turns written to results/tables/judge_hf_olmo-2-1b.jsonl
INFO Prompt SHA-256: 047a96d300b1d7d3bf97e302ea3c7285ea94871cbd2e887d6b148ca6ccea8da4
INFO Freeze this hash in configs/experiment.yaml before rerunning.


In [5]:
import pandas as pd, glob
for f in sorted(glob.glob("results/tables/judge_hf_*.jsonl")):
    d = pd.read_json(f, lines=True)
    print(f, len(d), d.status.value_counts().to_dict())
!zip -qr judge_results.zip results/tables/judge_hf_*.jsonl
files.download("judge_results.zip")

results/tables/judge_hf_olmo-2-1b.jsonl 400 {'ok': 400}
results/tables/judge_hf_qwen2.5-3b.jsonl 400 {'ok': 400}
results/tables/judge_hf_smollm2-1.7b.jsonl 400 {'ok': 400}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>